In [1]:
import sys
import nilmtk

print("Python:", sys.executable)
print("NILMTK:", nilmtk.__version__)

Python: c:\Users\LENOVO\Desktop\NILM\.venv\Scripts\python.exe
NILMTK: 0.4.1


In [2]:
from nilmtk import DataSet
import pandas as pd
import numpy as np

DATASET_PATH = "../data/processed/ukdale.h5"

ds = DataSet(DATASET_PATH)

print("Dataset loaded successfully.")
print("Buildings:", list(ds.buildings.keys()))

Dataset loaded successfully.
Buildings: [1, 2, 3, 4, 5]


In [9]:
rows = []

for house_id, building in ds.buildings.items():
    for meter in building.elec.meters:

        appliances = meter.appliances

        if appliances:
            appliance_names = ", ".join(
                str(appliance)
                for appliance in appliances
            )
        else:
            appliance_names = "Unknown"

        rows.append({
            "House": house_id,
            "Meter": meter.instance(),
            "Appliance(s)": appliance_names,
            "Site Meter": meter.is_site_meter()
        })

meter_df = pd.DataFrame(rows)

meter_df = meter_df.sort_values(
    ["House", "Meter"]
).reset_index(drop=True)

meter_df

,House,Meter,Appliance(s),Site Meter
0,1,2,"Appliance(type='boiler', instance=1)",False
1,1,3,Appliance(type='solar thermal pumping station'...,False
2,1,4,"Appliance(type='laptop computer', instance=1),...",False
3,1,5,"Appliance(type='washer dryer', instance=1), Ap...",False
4,1,6,"Appliance(type='dish washer', instance=1)",False
...,...,...,...,...
103,5,22,"Appliance(type='dish washer', instance=1)",False
104,5,23,"Appliance(type='microwave', instance=1)",False
105,5,24,"Appliance(type='washer dryer', instance=1)",False
106,5,25,"Appliance(type='vacuum cleaner', instance=1)",False


In [7]:
# Inspect the appliance metadata of House 1, Meter 4

meter = ds.buildings[1].elec.meters[4]

print("Meter:", meter)
print("\nAppliances:")
print(meter.appliances)

Meter: ElecMeter(instance=6, building=1, dataset='UK-DALE', appliances=[Appliance(type='dish washer', instance=1)])

Appliances:
[Appliance(type='dish washer', instance=1)]


In [8]:
print(type(meter.appliances))

if meter.appliances:
    print(type(meter.appliances[0]))
    print(meter.appliances[0])

<class 'list'>
<class 'nilmtk.appliance.Appliance'>
Appliance(type='dish washer', instance=1)


In [10]:
# Check the available measurement types for each site meter

for house_id, building in ds.buildings.items():
    site_meter = building.elec.mains()

    print(f"\nHouse {house_id}")
    print("Site meter:", site_meter)

    try:
        print("Available measurements:")
        print(site_meter.available_measurements())
    except Exception as e:
        print("Could not read measurements:", e)


House 1
Site meter: ElecMeter(instance=54, building=1, dataset='UK-DALE', site_meter, appliances=[Appliance(type='immersion heater', instance=1), Appliance(type='water pump', instance=1), Appliance(type='security alarm', instance=1), Appliance(type='fan', instance=2), Appliance(type='drill', instance=1), Appliance(type='laptop computer', instance=2)])
Available measurements:
Could not read measurements: 'ElecMeter' object has no attribute 'available_measurements'

House 2
Site meter: ElecMeter(instance=20, building=2, dataset='UK-DALE', site_meter, appliances=[])
Available measurements:
Could not read measurements: 'ElecMeter' object has no attribute 'available_measurements'

House 3
Site meter: ElecMeter(instance=1, building=3, dataset='UK-DALE', site_meter, appliances=[])
Available measurements:
Could not read measurements: 'ElecMeter' object has no attribute 'available_measurements'

House 4
Site meter: ElecMeter(instance=1, building=4, dataset='UK-DALE', site_meter, appliances=[])

In [11]:
# Inspect the actual data stored in each house's site meter

for house_id, building in ds.buildings.items():
    site_meter = building.elec.mains()

    print(f"\n{'='*60}")
    print(f"House {house_id}")
    print(site_meter)

    try:
        chunk = next(site_meter.load())

        print("\nColumns:")
        print(chunk.columns)

        print("\nNumber of samples:", len(chunk))

        print("\nFirst 5 rows:")
        display(chunk.head())

        print("\nLast 5 rows:")
        display(chunk.tail())

        print("\nTime range:")
        print("Start:", chunk.index.min())
        print("End:  ", chunk.index.max())

    except Exception as e:
        print("Could not load data:", e)


House 1
ElecMeter(instance=54, building=1, dataset='UK-DALE', site_meter, appliances=[Appliance(type='immersion heater', instance=1), Appliance(type='water pump', instance=1), Appliance(type='security alarm', instance=1), Appliance(type='fan', instance=2), Appliance(type='drill', instance=1), Appliance(type='laptop computer', instance=2)])

Columns:
MultiIndex([(  'power',   'active'),
            (  'power', 'apparent'),
            ('voltage',         '')],
           names=['physical_quantity', 'type'])

Number of samples: 128237371

First 5 rows:


physical_quantity                         power                 voltage
type                                     active    apparent            
2013-03-17 19:12:43.099999905+00:00  337.880005  431.040009  240.149994
2013-03-17 19:12:44.099999905+00:00  339.429993  427.940002  240.559998
2013-03-17 19:12:45.099999905+00:00  340.630005  429.660004  241.070007
2013-03-17 19:12:46.099999905+00:00  338.799988  426.989990  240.440002
2013-03-17 19:12:47.099999905+00:00  340.880005  429.130005  241.009995


Last 5 rows:


physical_quantity                         power                 voltage
type                                     active    apparent            
2017-04-26 18:35:54.599999905+01:00  592.419983  649.909973  245.490005
2017-04-26 18:35:55.500000+01:00     575.109985  633.219971  245.539993
2017-04-26 18:35:56.599999905+01:00  575.270020  636.380005  245.539993
2017-04-26 18:35:57.500000+01:00     582.239990  646.119995  245.470001
2017-04-26 18:35:58.500000+01:00     586.840027  647.799988  245.429993


Time range:
Start: 2013-03-17 19:12:43.099999905+00:00
End:   2017-04-26 18:35:58.500000+01:00

House 2
ElecMeter(instance=20, building=2, dataset='UK-DALE', site_meter, appliances=[])

Columns:
MultiIndex([(  'power',   'active'),
            (  'power', 'apparent'),
            ('voltage',         '')],
           names=['physical_quantity', 'type'])

Number of samples: 12166699

First 5 rows:


...


Time range:
Start: 2013-03-09 14:40:07+00:00
End:   2013-10-01 06:15:07+01:00

House 5
ElecMeter(instance=26, building=5, dataset='UK-DALE', site_meter, appliances=[])

Columns:
MultiIndex([(  'power',   'active'),
            (  'power', 'apparent'),
            ('voltage',         '')],
           names=['physical_quantity', 'type'])

Number of samples: 11405812

First 5 rows:


physical_quantity                         power                 voltage
type                                     active    apparent            
2014-06-29 17:23:43.200000048+01:00  702.570007  805.690002  243.690002
2014-06-29 17:23:44.299999952+01:00  697.150024  803.020020  243.610001
2014-06-29 17:23:45.299999952+01:00  689.039978  797.650024  243.619995
2014-06-29 17:23:46.299999952+01:00  693.409973  801.119995  243.710007
2014-06-29 17:23:47.299999952+01:00  701.049988  803.489990  243.750000


Last 5 rows:


physical_quantity                         power                 voltage
type                                     active    apparent            
2014-11-13 20:35:21.099999905+00:00  598.960022  721.000000  247.720001
2014-11-13 20:35:22.099999905+00:00  599.719971  722.419983  247.539993
2014-11-13 20:35:23.099999905+00:00  592.880005  715.280029  247.550003
2014-11-13 20:35:24.099999905+00:00  597.000000  720.030029  247.839996
2014-11-13 20:35:25.099999905+00:00  596.869995  719.219971  248.009995


Time range:
Start: 2014-06-29 17:23:43.200000048+01:00
End:   2014-11-13 20:35:25.099999905+00:00


In [12]:
# ============================================================
# UK-DALE SITE METER DATA PROFILE
# ============================================================

profile = []

for house_id, building in ds.buildings.items():

    site_meter = building.elec.mains()

    print(f"Profiling House {house_id}...")

    try:
        chunk = next(site_meter.load())

        # Timestamp information
        timestamps = chunk.index

        start_time = timestamps.min()
        end_time = timestamps.max()

        # Calculate timestamp differences in seconds
        time_diffs = timestamps.to_series().diff().dt.total_seconds().dropna()

        # Most common sampling interval
        median_interval = time_diffs.median()
        mode_interval = time_diffs.mode().iloc[0] if not time_diffs.mode().empty else np.nan

        # Duplicate timestamps
        duplicate_count = timestamps.duplicated().sum()

        # Gaps larger than 2x the median interval
        large_gap_count = (time_diffs > 2 * median_interval).sum()

        # Total duration
        duration_days = (end_time - start_time).total_seconds() / (24 * 3600)

        profile.append({
            "House": house_id,
            "Samples": len(chunk),
            "Start": start_time,
            "End": end_time,
            "Duration (days)": round(duration_days, 2),
            "Median interval (s)": round(median_interval, 4),
            "Mode interval (s)": round(mode_interval, 4),
            "Duplicate timestamps": int(duplicate_count),
            "Large gaps": int(large_gap_count),
            "Measurements": ", ".join(
                str(col) for col in chunk.columns
            )
        })

    except Exception as e:

        print(f"Error in House {house_id}: {e}")

profile_df = pd.DataFrame(profile)

profile_df

Profiling House 1...
Profiling House 2...
Profiling House 3...
Profiling House 4...
Profiling House 5...


,House,Samples,Start,End,Duration (days),Median interval (s),Mode interval (s),Duplicate timestamps,Large gaps,Measurements
0,1,128237371,2013-03-17 19:12:43.099999905+00:00,2017-04-26 18:35:58.500000+01:00,1500.93,1.0,1.0,0,1355,"('power', 'active'), ('power', 'apparent'), ('..."
1,2,12166699,2013-04-16 21:45:16.299999952+01:00,2013-10-10 06:16:00.500000+01:00,176.35,1.0,1.0,0,12,"('power', 'active'), ('power', 'apparent'), ('..."
2,3,512327,2013-02-27 20:35:14+00:00,2013-04-08 06:14:53+01:00,39.36,6.0,6.0,0,725,"('power', 'apparent')"
3,4,2186446,2013-03-09 14:40:07+00:00,2013-10-01 06:15:07+01:00,205.61,6.0,6.0,0,4423,"('power', 'apparent')"
4,5,11405812,2014-06-29 17:23:43.200000048+01:00,2014-11-13 20:35:25.099999905+00:00,137.17,1.0,1.0,0,6966,"('power', 'active'), ('power', 'apparent'), ('..."


In [13]:
# ============================================================
# GAP ANALYSIS FOR UK-DALE SITE METERS
# ============================================================

gap_results = []

for house_id, building in ds.buildings.items():

    site_meter = building.elec.mains()

    print(f"Analyzing House {house_id}...")

    chunk = next(site_meter.load())

    timestamps = chunk.index

    # Calculate differences between consecutive timestamps
    time_diffs = (
        timestamps.to_series()
        .diff()
        .dt.total_seconds()
        .dropna()
    )

    median_interval = time_diffs.median()

    # Gaps larger than 2x the normal sampling interval
    gaps = time_diffs[time_diffs > 2 * median_interval]

    # Categorize gaps
    gap_results.append({
        "House": house_id,
        "Median interval (s)": median_interval,
        "Total gaps > 2x": len(gaps),
        "Gaps > 10s": (time_diffs > 10).sum(),
        "Gaps > 1 min": (time_diffs > 60).sum(),
        "Gaps > 10 min": (time_diffs > 600).sum(),
        "Gaps > 1 hour": (time_diffs > 3600).sum(),
        "Largest gap (s)": time_diffs.max(),
        "Largest gap (hours)": time_diffs.max() / 3600
    })

gap_df = pd.DataFrame(gap_results)

gap_df

Analyzing House 1...
Analyzing House 2...
Analyzing House 3...
Analyzing House 4...
Analyzing House 5...


,House,Median interval (s),Total gaps > 2x,Gaps > 10s,Gaps > 1 min,Gaps > 10 min,Gaps > 1 hour,Largest gap (s),Largest gap (hours)
0,1,1.0,1355,71,21,12,9,884955.6,245.821000
1,2,1.0,12,12,9,5,3,2968489.4,824.580389
2,3,6.0,725,1053,120,19,7,131107.0,36.418611
3,4,6.0,4423,6719,5,2,1,4294974.0,1193.048333
4,5,1.0,6966,5,5,5,2,402775.6,111.882111


In [14]:
# ============================================================
# TIMESTAMP GAP DISTRIBUTION
# ============================================================

for house_id, building in ds.buildings.items():

    site_meter = building.elec.mains()
    chunk = next(site_meter.load())

    timestamps = chunk.index

    time_diffs = (
        timestamps.to_series()
        .diff()
        .dt.total_seconds()
        .dropna()
    )

    print(f"\n{'='*60}")
    print(f"House {house_id}")

    print("\nBasic statistics of timestamp intervals:")
    print(time_diffs.describe())

    print("\nMost common timestamp intervals:")
    print(
        time_diffs
        .round(3)
        .value_counts()
        .head(15)
    )


House 1

Basic statistics of timestamp intervals:
count    1.282374e+08
mean     1.011254e+00
std      8.641632e+01
min     -7.751550e+04
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      8.849556e+05
dtype: float64

Most common timestamp intervals:
1.0    101541507
0.9     13733429
1.1     12955942
0.1         1279
1.2          880
0.8          342
1.3          336
0.7          294
0.2          247
0.5          214
0.6          214
0.3          204
1.4          200
0.4          195
1.5          177
Name: count, dtype: int64

House 2

Basic statistics of timestamp intervals:
count    1.216670e+07
mean     1.252357e+00
std      8.514219e+02
min     -1.700000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      2.968489e+06
dtype: float64

Most common timestamp intervals:
1.0       9640400
0.9       1295476
1.1       1230679
1.2            99
1.4             7
0.8             6
0.7             6
1.3             6
1.5             2
0.5  

In [15]:
# ============================================================
# FIND OUT-OF-ORDER TIMESTAMPS
# ============================================================

for house_id, building in ds.buildings.items():

    site_meter = building.elec.mains()
    chunk = next(site_meter.load())

    timestamps = chunk.index

    time_diffs = (
        timestamps.to_series()
        .diff()
        .dt.total_seconds()
    )

    negative_gaps = time_diffs[time_diffs < 0]

    print(f"\n{'='*60}")
    print(f"House {house_id}")
    print("Number of negative intervals:", len(negative_gaps))

    if len(negative_gaps) > 0:

        print("\nNegative intervals:")
        print(negative_gaps)

        print("\nLocations:")
        for idx in negative_gaps.index:
            position = timestamps.get_loc(idx)

            print("\nTimestamp before:")
            print(timestamps[position - 1])

            print("Timestamp after:")
            print(timestamps[position])

            print("Difference (seconds):")
            print(time_diffs.loc[idx])


House 1
Number of negative intervals: 1

Negative intervals:
2013-04-10 15:39:51.500000+01:00   -77515.5
dtype: float64

Locations:

Timestamp before:
2013-04-11 13:11:47+01:00
Timestamp after:
2013-04-10 15:39:51.500000+01:00
Difference (seconds):
-77515.5

House 2
Number of negative intervals: 1

Negative intervals:
2013-08-06 07:54:14+01:00   -1.7
dtype: float64

Locations:

Timestamp before:
2013-08-06 07:54:15.700000048+01:00
Timestamp after:
2013-08-06 07:54:14+01:00
Difference (seconds):
-1.700000048

House 3
Number of negative intervals: 0

House 4
Number of negative intervals: 0

House 5
Number of negative intervals: 0


In [16]:
# ============================================================
# HOUSE 1 APPLIANCE TIMESTAMP CHECK
# ============================================================

appliances_to_check = {
    "Kettle / Food processor / Sandwich maker": 10,
    "Fridge freezer": 12,
    "Washer dryer": 5
}

for appliance_name, meter_id in appliances_to_check.items():

    meter = ds.buildings[1].elec.meters[meter_id]

    print(f"\n{'='*70}")
    print(f"{appliance_name}")
    print(meter)

    try:
        chunk = next(meter.load())

        timestamps = chunk.index

        time_diffs = (
            timestamps.to_series()
            .diff()
            .dt.total_seconds()
            .dropna()
        )

        print("\nColumns:")
        print(chunk.columns)

        print("Samples:", len(chunk))
        print("Start:", timestamps.min())
        print("End:", timestamps.max())

        print("\nTimestamp interval:")
        print("Median:", time_diffs.median(), "seconds")
        print("Minimum:", time_diffs.min(), "seconds")
        print("Maximum:", time_diffs.max(), "seconds")

        print("Negative intervals:", (time_diffs < 0).sum())
        print("Duplicate timestamps:", timestamps.duplicated().sum())

    except Exception as e:
        print("Could not load:", e)


Kettle / Food processor / Sandwich maker
ElecMeter(instance=12, building=1, dataset='UK-DALE', appliances=[Appliance(type='fridge freezer', instance=1)])

Columns:
MultiIndex([('power', 'active')],
           names=['physical_quantity', 'type'])
Samples: 19381298
Start: 2012-12-14 22:21:32+00:00
End: 2017-04-26 18:32:51+01:00

Timestamp interval:
Median: 7.0 seconds
Minimum: 3.0 seconds
Maximum: 626776.0 seconds
Negative intervals: 0
Duplicate timestamps: 0

Fridge freezer
ElecMeter(instance=14, building=1, dataset='UK-DALE', appliances=[Appliance(type='computer monitor', instance=1)])

Columns:
MultiIndex([('power', 'active')],
           names=['physical_quantity', 'type'])
Samples: 4143648
Start: 2012-12-14 22:21:33+00:00
End: 2017-04-26 18:33:09+01:00

Timestamp interval:
Median: 7.0 seconds
Minimum: 3.0 seconds
Maximum: 1896973.0 seconds
Negative intervals: 0
Duplicate timestamps: 0

Washer dryer
ElecMeter(instance=7, building=1, dataset='UK-DALE', appliances=[Appliance(type='tel

In [17]:
def get_meter_by_instance(building, instance_number):

    for meter in building.elec.meters:

        if meter.instance() == instance_number:
            return meter

    return None

In [18]:
meter = get_meter_by_instance(ds.buildings[1], 12)

print(meter)

ElecMeter(instance=12, building=1, dataset='UK-DALE', appliances=[Appliance(type='fridge freezer', instance=1)])


In [19]:
# ============================================================
# HOUSE 1: ACTUAL METER INSTANCE → APPLIANCE MAPPING
# ============================================================

building = ds.buildings[1]

rows = []

for meter in building.elec.meters:

    appliances = meter.appliances

    if appliances:

        appliance_names = ", ".join(
            str(appliance)
            for appliance in appliances
        )

    else:
        appliance_names = "Unknown"

    rows.append({
        "Meter Instance": meter.instance(),
        "Appliance(s)": appliance_names,
        "Site Meter": meter.is_site_meter()
    })

h1_meter_df = pd.DataFrame(rows)

h1_meter_df

,Meter Instance,Appliance(s),Site Meter
0,2,"Appliance(type='boiler', instance=1)",False
1,3,Appliance(type='solar thermal pumping station'...,False
2,4,"Appliance(type='laptop computer', instance=1),...",False
3,5,"Appliance(type='washer dryer', instance=1), Ap...",False
4,6,"Appliance(type='dish washer', instance=1)",False
5,7,"Appliance(type='television', instance=1)",False
6,8,"Appliance(type='light', instance=1), Appliance...",False
7,9,"Appliance(type='HTPC', instance=1)",False
8,10,"Appliance(type='kettle', instance=1), Applianc...",False
9,11,"Appliance(type='toaster', instance=1), Applian...",False
